In [35]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
from datetime import datetime
import re
from tqdm import tqdm

In [36]:
def get_match_details(link):
    headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/102.0.0.0 Safari/537.36'}
    res = requests.get(link)
    soup = BeautifulSoup(res.content)
    date = ""
    attendance = ""
    for elem in soup.find_all(name = 'div', attrs = {"class":"mc-summary__info"}):
        try:
            date = datetime.strptime(elem.text.strip(), "%a %d %b %Y")
        except:
            pass
        try:
            if "Att" in elem.text:
                attendance = elem.text.strip()
                attendance = int(attendance.replace("Att:", "").strip().replace(',', ''))
        except:
            pass
        
    home_team, away_team = list(map(lambda x: x.find_all(name = "span", attrs = {'u-hide-phablet'})[0].text, soup.find_all(name = 'div', attrs = {"class":"mc-summary__team"})))
    home_team_goals, away_team_goals = list(map(int, soup.find_all(name = 'div', attrs = {"class":"mc-summary__score"})[0].text.split(' - ')))
    
    home_team_goals_half_time, away_team_goals_half_time = map(int, soup.find_all(name = 'div', attrs = {"class":"mc-summary__half-time"})[0].findChild("span").text.split('-'))
    
    event_list = []
    for elem in soup.find_all(name = 'div', attrs = {"class":"timeLineEventsContainer"}
                             )[0].findChildren(name = "div", attrs = {'class' : 'event'}):
        for list_elements in elem.findChildren("ul", attrs = {'class' : 'event__icons'}):
            for team in list_elements.get_attribute_list("class"):
                if "--" in team:
                    team = team.split('--')[-1]
                    break
            for events in list_elements.findChildren("li", attrs = {'class' : 'event__icon'}):
                if "event__icon--dummy" in events.get_attribute_list("class"):
                    continue
                try:
                    header = events.findChildren("div")[0]
                    header = header.findChildren("div")[0]
                    header = header.findChildren("header")[0]
                    time = header.findChildren("time")[0].text
                    event = header.findChildren("span")[0].text
                    event_list.append((team, time, event))
                except:
                    pass
    return {
        'match_date' : date,
        'attendance' : attendance,
        'home_team' : home_team,
        'away_team' : away_team,
        'home_team_goals' : home_team_goals,
        'away_team_goals' : away_team_goals,
        'home_team_goals_half_time' : home_team_goals_half_time,
        'away_team_goals_half_time' : away_team_goals_half_time,
        'events' : event_list
    }

In [37]:
matches = list(range(66342, 66722))
match_data = []
for match in tqdm(matches):
    link = f"https://www.premierleague.com/match/{match}"
    try:
        dic = get_match_details(link)
        dic['link'] = link
    except Exception as e:
        print("Exception", str(e), "\n", link)
        dic = {
        'match_date' : "",
        'attendance' : 0,
        'home_team' : "",
        'away_team' : "",
        'home_team_goals' : "",
        'away_team_goals' : "",
        'home_team_goals_half_time' : "",
        'away_team_goals_half_time' : "",
        'events' : [],
        "link" : link
    }
    match_data.append(dic)

100%|████████████████████████████████████████████████████████████████████████████████| 380/380 [08:05<00:00,  1.28s/it]


In [38]:
df = pd.DataFrame(match_data)

In [39]:
def minute_parser(time_string):
    time_string = time_string.replace("'",'').replace('"', '')
    if "90 +" in time_string or "45 +" in time_string:
#         print(time_string)
        return int(time_string[:2]) + int(time_string[4:])
    return int(time_string)

In [40]:
df['events'] = df.events.apply(lambda x: [(i[0], minute_parser(i[1]), i[2]) for i in x])

In [41]:
df.head()

,match_date,attendance,home_team,away_team,home_team_goals,away_team_goals,home_team_goals_half_time,away_team_goals_half_time,events,link
0,2021-08-13,16479,Brentford,Arsenal,2,0,1,0,"[(home, 22, Goal), (away, 59, Substitution), (...",https://www.premierleague.com/match/66342
1,2021-08-14,16910,Burnley,Brighton,1,2,1,0,"[(home, 2, Goal), (home, 9, Yellow Card), (hom...",https://www.premierleague.com/match/66343
2,2021-08-14,38965,Chelsea,Crystal Palace,3,0,2,0,"[(home, 27, Goal), (home, 40, Goal), (away, 57...",https://www.premierleague.com/match/66344
3,2021-08-14,38487,Everton,Southampton,3,1,0,1,"[(away, 22, Goal), (home, 30, Yellow Card), (h...",https://www.premierleague.com/match/66345
4,2021-08-14,31983,Leicester,Wolves,1,0,1,0,"[(home, 41, Goal), (away, 59, Yellow Card), (h...",https://www.premierleague.com/match/66346


In [42]:
from collections import Counter

In [43]:
events_processed = df['events'].apply(lambda x: Counter([i[2] for i in x if i[1] >= 88])).apply(pd.Series).fillna(0).astype(int)

In [44]:
events_processed['match_date'] = df['match_date']
events_processed['link'] = df['link']

In [45]:
events_processed

,Substitution,Yellow Card,label.penalty.scored,Goal,Second Yellow Card (Red Card),Own Goal,Red Card,Substitution Off,match_date,link
0,0,0,0,0,0,0,0,0,2021-08-13,https://www.premierleague.com/match/66342
1,0,0,0,0,0,0,0,0,2021-08-14,https://www.premierleague.com/match/66343
2,0,0,0,0,0,0,0,0,2021-08-14,https://www.premierleague.com/match/66344
3,1,0,0,0,0,0,0,0,2021-08-14,https://www.premierleague.com/match/66345
4,1,1,0,0,0,0,0,0,2021-08-14,https://www.premierleague.com/match/66346
...,...,...,...,...,...,...,...,...,...,...
375,0,1,0,0,0,0,0,0,2022-05-22,https://www.premierleague.com/match/66717
376,1,0,0,1,0,0,0,0,2022-05-22,https://www.premierleague.com/match/66718
377,1,0,0,1,0,0,0,0,2022-05-22,https://www.premierleague.com/match/66719
378,1,0,0,0,0,0,0,0,2022-05-22,https://www.premierleague.com/match/66720


In [46]:
events_processed.to_csv('2021-22_events_after_88min.csv', index = False)

In [47]:
df.head()

,match_date,attendance,home_team,away_team,home_team_goals,away_team_goals,home_team_goals_half_time,away_team_goals_half_time,events,link
0,2021-08-13,16479,Brentford,Arsenal,2,0,1,0,"[(home, 22, Goal), (away, 59, Substitution), (...",https://www.premierleague.com/match/66342
1,2021-08-14,16910,Burnley,Brighton,1,2,1,0,"[(home, 2, Goal), (home, 9, Yellow Card), (hom...",https://www.premierleague.com/match/66343
2,2021-08-14,38965,Chelsea,Crystal Palace,3,0,2,0,"[(home, 27, Goal), (home, 40, Goal), (away, 57...",https://www.premierleague.com/match/66344
3,2021-08-14,38487,Everton,Southampton,3,1,0,1,"[(away, 22, Goal), (home, 30, Yellow Card), (h...",https://www.premierleague.com/match/66345
4,2021-08-14,31983,Leicester,Wolves,1,0,1,0,"[(home, 41, Goal), (away, 59, Yellow Card), (h...",https://www.premierleague.com/match/66346


In [48]:
df.to_csv('2021-22_data.csv', index = False)

In [49]:
all_events_processed = df['events'].apply(lambda x: Counter([i[2] for i in x])).apply(pd.Series).fillna(0).astype(int)

In [50]:
all_events_processed['match_date'] = df['match_date']
all_events_processed['link'] = df['link']

In [51]:
all_events_processed.to_csv('2021-22_all_events.csv', index = False)